In [2]:
%%bash
pip install -q uv
uv venv /content/lfenv --python 3.12 -q
uv pip install --python /content/lfenv/bin/python -q langflow
/content/lfenv/bin/langflow --version

langflow 1.13.0


agent_card passed to DefaultRequestHandlerV2: AgentCard is not spec-compliant - REQUIRED fields missing or empty: name, description, supported_interfaces, version, default_input_modes, default_output_modes, skills. This is allowed and does not raise, but it may not verify or interoperate across SDKs and could be rejected in a future major release. See https://a2a-protocol.org/latest/specification/#57-field-presence-and-optionality


In [1]:
USE_OLLAMA     = True
OLLAMA_MODEL   = "qwen2.5:7b"

USE_EXPLABS      = True
EXPLABS_BASE_URL = "https://api.experientiallabs.ai/v1"
EXPLABS_MODEL    = "deepseek-v4-flash"

LANGFLOW_PASSWORD = "123456"

EXPLABS_API_KEY = ""
try:
    from google.colab import userdata
    EXPLABS_API_KEY = userdata.get("EXPLABS_API_KEY") or ""
except Exception:
    pass
print("Experiential Labs key loaded:", bool(EXPLABS_API_KEY))

Experiential Labs key loaded: True


In [2]:
import subprocess, time, os
if USE_OLLAMA:
    !apt-get -qq install -y zstd pciutils lshw > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh
    subprocess.Popen(["ollama", "serve"],
                     stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT,
                     env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"})
    time.sleep(5)
    !ollama pull {OLLAMA_MODEL}
    !ollama list

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

NAME          ID              SIZE      MODIFIED               
qwen2.5:7b    845dbda0ea48    4.7 GB    Less than a second ago    


In [3]:
import requests

if USE_EXPLABS and EXPLABS_API_KEY:
    h = {"Authorization": f"Bearer {EXPLABS_API_KEY}"}
    r = requests.get(f"{EXPLABS_BASE_URL}/models", headers=h, timeout=30)
    print(r.status_code)
    ids = [m["id"] for m in r.json().get("data", [])]
    print(f"{len(ids)} models, first 50:", ids[:50])
    if EXPLABS_MODEL:
        r = requests.post(f"{EXPLABS_BASE_URL}/chat/completions", headers=h, timeout=60,
                          json={"model": EXPLABS_MODEL,
                                "messages": [{"role": "user", "content": "Say hi in 5 words"}]})
        print(r.status_code, r.text[:500])

401
0 models, first 50: []
401 {"error":{"message":"The gateway key is invalid, expired, or revoked. Ask the gateway operator to issue a new virtual key.","type":"authentication_error","param":null,"code":"invalid_key"}}


In [4]:
import secrets, requests, subprocess, time, os

env = {**os.environ,
       "LANGFLOW_AUTO_LOGIN": "False",
       "LANGFLOW_SUPERUSER": "langflow",
       "LANGFLOW_SUPERUSER_PASSWORD": LANGFLOW_PASSWORD,
       "LANGFLOW_SECRET_KEY": secrets.token_urlsafe(32)}

if EXPLABS_API_KEY:
    env["EXPLABS_API_KEY"] = EXPLABS_API_KEY
    env["LANGFLOW_VARIABLES_TO_GET_FROM_ENVIRONMENT"] = "EXPLABS_API_KEY"

lf = subprocess.Popen(["/content/lfenv/bin/langflow", "run", "--host", "0.0.0.0", "--port", "7860"],
                      stdout=open("/content/langflow.log", "w"), stderr=subprocess.STDOUT, env=env)
for _ in range(120):
    try:
        if requests.get("http://127.0.0.1:7860/health", timeout=2).ok:
            print("Langflow is running"); break
    except Exception:
        pass
    time.sleep(5)
else:
    print(open("/content/langflow.log").read()[-3000:])

Langflow is running


In [5]:
import os, time, requests, subprocess

os.makedirs("/content/lf_components/experiential_labs", exist_ok=True)
with open("/content/lf_components/experiential_labs/explabs_model.py", "w") as f:
    f.write('''
try:
    from lfx.base.models.model import LCModelComponent
    from lfx.field_typing import LanguageModel
    from lfx.io import MessageTextInput, SecretStrInput, FloatInput
except ImportError:
    from langflow.base.models.model import LCModelComponent
    from langflow.field_typing import LanguageModel
    from langflow.io import MessageTextInput, SecretStrInput, FloatInput
from langchain_openai import ChatOpenAI


class ExperientialLabsModel(LCModelComponent):
    display_name = "Experiential Labs"
    description = "Chat model via the Experiential Labs gateway (OpenAI-compatible)"
    icon = "OpenAI"

    inputs = [
        *LCModelComponent._base_inputs,
        MessageTextInput(name="base_url", display_name="Base URL",
                         value="https://api.experientiallabs.ai/v1", advanced=True),
        MessageTextInput(name="model_name", display_name="Model ID", value="''' + EXPLABS_MODEL + '''"),
        SecretStrInput(name="api_key", display_name="API Key", value="EXPLABS_API_KEY"),
        FloatInput(name="temperature", display_name="Temperature", value=0.3, advanced=True),
    ]

    def build_model(self) -> LanguageModel:
        return ChatOpenAI(model=self.model_name, base_url=self.base_url,
                          api_key=self.api_key, temperature=self.temperature)
''')

lf.terminate(); time.sleep(5)
env["LANGFLOW_COMPONENTS_PATH"] = "/content/lf_components"
lf = subprocess.Popen(["/content/lfenv/bin/langflow", "run", "--host", "0.0.0.0", "--port", "7860"],
                      stdout=open("/content/langflow.log", "w"), stderr=subprocess.STDOUT, env=env)
for _ in range(120):
    try:
        if requests.get("http://127.0.0.1:7860/health", timeout=2).ok:
            print("Langflow restarted with the Experiential Labs component"); break
    except Exception:
        pass
    time.sleep(5)
else:
    print(open("/content/langflow.log").read()[-3000:])

Langflow restarted with the Experiential Labs component


In [6]:
import re, subprocess, time

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

cf = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:7860", "--no-autoupdate"],
                      stdout=open("/content/cf.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cf.log").read())
    if m:
        url = m.group(0); break
    time.sleep(1)
print("Open Langflow here:", url)
print("Login → user: langflow | password:", LANGFLOW_PASSWORD)

Open Langflow here: https://pack-basket-wrestling-earned.trycloudflare.com
Login → user: langflow | password: 123456
